In [4]:
# Import delle librerie
import numpy as np
import dill

# Caricamento dello spazio di lavoro
with open('Spazio_di_Lavoro_Medori.pkl', 'rb') as file:
    reachable_points = dill.load(file)
# Caricamento funzionale (distanza dal fine corsa)
with open('W_q_numpy_functions_fine_corsa.pkl', 'rb') as file:
    dw_dq_numpy = dill.load(file)
dw_dq_func = dw_dq_numpy

# Lunghezza link
l0 = 0.11
l1 = 0.065
l2 = 0.065
l3 = 0.03
d3 = 0.03
d4 = 0.105

# Parametri del manipolatore 
a = np.array([l1, l2, 0.0, l3])
q0 = np.array([0, 0, 0, 0])
d = np.array([0, 0, -d3, d4])
alpha = np.array([-np.pi/2, np.pi/2, -np.pi/2, 0])
#  Funzione trasformazione DH 
def DH(d, a, alpha, theta):
    T = np.array([
        [np.cos(theta), -np.sin(theta)*np.cos(alpha), np.sin(theta)*np.sin(alpha), a*np.cos(theta)],
        [np.sin(theta), np.cos(theta)*np.cos(alpha), -np.cos(theta)*np.sin(alpha), a*np.sin(theta)],
        [0.0,           np.sin(alpha),               np.cos(alpha),               d],
        [0.0,           0.0,                         0.0,                         1.0]
    ])
    return T

Tb0 = DH(0, 0, np.pi/2, 0)

# Jacobiano 
def Jacobian(q):
    # Trasformazioni DH relative tra giunti
    T01 = DH(d[0], a[0], alpha[0], q[0])
    T12 = DH(d[1], a[1], alpha[1], q[1])
    T23 = DH(d[2], a[2], alpha[2], q[2])
    T34 = DH(d[3], a[3], alpha[3], q[3])

    T0 = Tb0     
    T1 = np.matmul(T0, T01)
    T2 = np.matmul(T1, T12)
    T3 = np.matmul(T2, T23)
    T4 = np.matmul(T3, T34)

    # Vettori z degli assi di rotazione
    z0 = T0[:3, 2]
    z1 = T1[:3, 2]
    z2 = T2[:3, 2]
    z3 = T3[:3, 2]

    # Vettori posizione dei giunti 
    p0 = T0[:3, 3]
    p1 = T1[:3, 3]
    p2 = T2[:3, 3]
    p3 = T3[:3, 3]
    p_ee = T4[:3, 3]

    # Jacobiano parte lineare (Jv)
    J = np.column_stack((
        np.cross(z0, p_ee - p0),
        np.cross(z1, p_ee - p1),
        np.cross(z2, p_ee - p2),
        np.cross(z3, p_ee - p3)
    ))

    return J

def dir_kin(Qd):
    T01 = DH(d[0], a[0], alpha[0], Qd[0])
    T12 = DH(d[1], a[1], alpha[1], Qd[1])
    T23 = DH(d[2], a[2], alpha[2], Qd[2])
    T34 = DH(d[3], a[3], alpha[3], Qd[3])

    Tb1 = np.matmul(Tb0, T01)
    Tb2 = np.matmul(Tb1, T12)
    Tb3 = np.matmul(Tb2, T23)
    Tb4 = np.matmul(Tb3, T34)

    p_1 = Tb1[:3, 3]
    p_2 = Tb2[:3, 3]
    p_3 = Tb3[:3, 3]
    p_ee = Tb4[:3, 3]

    joint_1 = p_1  
    joint_2 = p_2
    corner_point = p_3
    joint_3 = p_3 + d4*Tb3[:3, 2]
    joint_4 = p_ee
    

    return p_1, p_2, p_3, p_ee, joint_1, joint_2, joint_3, joint_4, corner_point

# Funzione per il Calcolo del Centro di un arco 
def center(p1, p2, radius):

    PM = (p1 + p2) / 2   # Calcola il punto medio tra p1 e p2
    p1p2 = p2 - p1    # Calcola il vettore che collega p1 a p2

    # Scegli un vettore ausiliario non parallelo a p1p2
    v = np.array([0, 0, 1]) 
    if np.allclose(np.cross(p1p2, v), 0):
        v = np.array([0, 1, 0])  # Se p1p2 è parallelo a Z, usa il vettore lungo Y

    # Calcola un vettore perpendicolare a p1p2 e v tramite il prodotto vettoriale
    perp_p1p2 = np.cross(p1p2, v)
    
    # Normalizza il vettore perpendicolare (diventa un vettore unitario)
    perp_p1p2 = perp_p1p2 / np.linalg.norm(perp_p1p2)

    # Calcolo la metà della lunghezza del segmento p1p2 
    half_p1p2_length = np.linalg.norm(p1p2) / 2

    # Se il raggio è troppo piccolo (minore della metà della distanza), non è possibile trovare un centro
    if radius <= half_p1p2_length:
        raise ValueError("Non è possibile trovare un centro con il raggio fornito")

    # Calcola la distanza tra il punto medio e il centro dell'arco
    dist = np.sqrt(radius**2 - half_p1p2_length**2)

    # Calcola il centro dell'arco come il punto medio più la distanza lungo il vettore perpendicolare
    C = PM + dist * perp_p1p2

    return C

# Funzione per pianificare un arco di circonferenza nello spazio operativo
def CircumferencePlanner(C, pi, pf, t0, tf, t, path):        
    rho = np.linalg.norm(pi - C)
    theta1 = 2 * np.arctan2(0.5 * np.linalg.norm(pf - pi), np.sqrt(rho**2 - (0.25 * np.linalg.norm(pf - pi)**2)))
    theta2 = 2 * np.pi - theta1
    short_arc = rho * min(theta1, theta2)
    long_arc = rho * max(theta1, theta2)

    if t <= t0:
        pd = pi
        pdot = np.zeros(3)
        
    elif t>=tf:
        pd = pf
        pdot = np.zeros(3)

    else :
        A = np.array([
            [t0**3, t0**2, t0, 1],
            [tf**3, tf**2, tf, 1],
            [ 3*t0**2, 2*t0, 1, 0],
            [ 3*tf**2, 2*tf, 1, 0],
        ])

        if path == "short":
            b = np.array([0, short_arc, 0, 0])
        elif path == "long":
            b = np.array([0, long_arc, 0, 0])
        else:
            raise ValueError("Percorso deve essere 'short' o 'long'.")
        
        coeff = np.linalg.solve(A, b)
        # Polinomio di terzo grado per calcolare la posizione (s)
        s = coeff[0]*t**3 + coeff[1]*t**2 + coeff[2]*t + coeff[3]

        # Derivate prima del polinomio (velocità)
        sdot = 3*coeff[0]*t**2 + 2*coeff[1]*t + coeff[2]

        pd1 = np.array([
            rho * np.cos(s / rho),
            rho * np.sin(s / rho),
            0
        ])

        pd1dot = np.array([
            -np.sin(s / rho) * sdot,
             np.cos(s / rho) * sdot,
             0
        ])

        x1 = (pi - C) / rho
        z1_vect = np.cross(x1, (pf - C) / rho)
        z1 = z1_vect / np.linalg.norm(z1_vect)
        if path == "long":
            z1 = -z1
        y1 = np.cross(z1, x1)
        R = np.column_stack([x1, y1, z1])
        
        pd = C + np.matmul(R, pd1)
        pdot = np.matmul(R, pd1dot)
    return pd, pdot

# Funzione per calcolarsi i limiti dello spazio di lavoro
def calculate_workspace_limits(workspace_points):
        x_min, x_max = float('inf'), -float('inf')
        y_min, y_max = float('inf'), -float('inf')
        z_min, z_max = float('inf'), -float('inf')

        for point in workspace_points:
            x, y, z = point
            # Aggiorno limiti con i valori minimi e massimi per tutti e 3 gli assi
            x_min = min(x_min, x)
            x_max = max(x_max, x)
            y_min = min(y_min, y)
            y_max = max(y_max, y)
            z_min = min(z_min, z)
            z_max = max(z_max, z)

        return (x_min, x_max), (y_min, y_max), (z_min, z_max)

# Funzione per vedere se un punto appartiente allo spazio di lavoro
def is_point_in_workspace(point, workspace_limits):
    # Definire i limiti dello spazio di lavoro del robot
    x_min, x_max = workspace_limits[0]  
    y_min, y_max = workspace_limits[1] 
    z_min, z_max = workspace_limits[2]  
    if (x_min <= point[0] <= x_max) and (y_min <= point[1] <= y_max) and (z_min <= point[2] <= z_max):
        return True
    else:
        return False 

def inv_kin(q, xd, xdot, k, method, ka, dw_dq_func, q_mean, q_max, q_min):
    # Calcoliamo lo Jacobiano J nella configurazione corrente q
    J = Jacobian(q)

    # Costruiamo le matrici DH per ciascun giunto e otteniamo Tb1–Tb4
    T01 = DH(d[0], a[0], alpha[0], q[0])
    T12 = DH(d[1], a[1], alpha[1], q[1])
    T23 = DH(d[2], a[2], alpha[2], q[2])
    T34 = DH(d[3], a[3], alpha[3], q[3])

    Tb1 = np.matmul(Tb0, T01)
    Tb2 = np.matmul(Tb1, T12)
    Tb3 = np.matmul(Tb2, T23)
    Tb4 = np.matmul(Tb3, T34)

    # Estraiamo la posizione attuale dell’end-effector
    p_ee = Tb4[:3, 3]

    # Calcoliamo l’errore di posizione e la sua norma
    e = xd - p_ee
    err = np.linalg.norm(e)

    # Termine di correzione per la velocità articolare
    K = k * np.eye(3)
    correction = xdot + np.matmul(K, e)

    # Inizializziamo Qdot e il contatore di iterazioni
    Qdot = np.zeros_like(q)

    if method == 'inverse':
        # Soluzione classica con pseudoinversa di J
        Qdot = np.matmul(np.linalg.pinv(J), correction)

    elif method == 'transpose':
        Qdot = np.matmul(J.T, correction)

    elif method == 'redundant':
        # Proiezione nello spazio nullo
        P = np.eye(4) - np.matmul(np.linalg.pinv(J), J)
        # Calcolo del gradiente dw/dq
        dw_dq_val = dw_dq_func(*np.concatenate([q, q_mean, q_max, q_min]))
        # Velocità articolare considerando la ridondanza
        Qdot = (np.matmul(np.linalg.pinv(J), correction) + ka * np.matmul(P, dw_dq_val))

    return Qdot, err

In [5]:
q0 = np.array([0,0,0,0])
# Calcolo matrice di trasformazione tra base e end-effector
T01 = T01 = DH(d[0], a[0], alpha[0], q0[0])
T12 = T12 = DH(d[1], a[1], alpha[1], q0[1])
T23 = T23 = DH(d[2], a[2], alpha[2], q0[2])
T34 = T34 = DH(d[3], a[3], alpha[3], q0[3])
Tb1 = np.matmul(Tb0,T01)
Tb2 = np.matmul(Tb1,T12)
Tb3 = np.matmul(Tb2,T23)
Tb4 = np.matmul(Tb3,T34)
pee = Tb4[:3,3]
print('Tb4 = \n',Tb4)
print('\npee = ',pee)
###################################
# PIANIFICAZIONE DELLA TRAIETORIA

# VISUALIZZAZIONE 3D DELLA TRAIETTORIA
# Scelta punti per la pianificazione di traiettoria e controllo se appartengono al mio workspace
_, _, _, pA, *_ = dir_kin(q0)
pB = np.array([0.1, 0.15, 0.13])
pC = np.array([-0.2, 0.05, 0.1])
print('Point A has coordinates : ', pA)
print('Point B has coordinates : ', pB)
print('Point C has coordinates : ', pC)
# Trovo limiti del mio workspace
workspace_limits = calculate_workspace_limits(reachable_points)
print(" Workspace limits:")
print("X:", workspace_limits[0])
print("Y:", workspace_limits[1])
print("Z:", workspace_limits[2])
if is_point_in_workspace(pA, workspace_limits) and is_point_in_workspace(pB,workspace_limits) and is_point_in_workspace(pC,workspace_limits) : 
    print('All the points selected for the planning belong to the workspace')
else:
    print('At least one of the selected points for the planning does NOT belong to the workspace')
C1 = center(pA, pB, 0.1)
C2 = center(pB, pC, 0.3)
print("C1", C1)
print("C2", C2)
# Pianifichiamo le traiettorie
tA = 0
tB = 10
tC = 20
path_1 = "short" 
path_2 = "short"  
t_values_1 = np.arange(tA, tB + 0.05, 0.05)
t_values_2 = np.arange(tB, tC + 0.05, 0.05)
# Preparo i dati per la visualizzazione
trajectory_1 = np.zeros((len(t_values_1), 3))
trajectory_2 = np.zeros((len(t_values_2), 3))
speed_1 = np.zeros((len(t_values_1), 3))
speed_2 = np.zeros((len(t_values_2), 3))
for i, t in enumerate(t_values_1):
    pd, pdot = CircumferencePlanner(C1, pA, pB, tA, tB, t, path_1)
    trajectory_1[i, :] = pd
    speed_1[i, :] = pdot
for i, t in enumerate(t_values_2):
    pd, pdot = CircumferencePlanner(C2, pB, pC, tB, tC, t, path_2)
    trajectory_2[i, :] = pd
    speed_2[i, :] = pdot
###################################
# INVERSIONE CINEMATICA
q0 = np.array([0, 0, 0, 0], dtype = float)
n_samples = 100
# Definisco range dei giunti per calcolare minimo, massimo e valore centrale
q1_range = np.linspace(0, 175*np.pi/180, n_samples) # da 0 a 175
q2_range = np.linspace(-14*np.pi/180, 106*np.pi/180, n_samples) # da -14 a 106
q3_range = np.linspace(-2*np.pi/3, 7*np.pi/10, n_samples) # da -120 a 126
q4_range = np.linspace(0, 2*np.pi, n_samples) # da 0 a 360
q_min = np.array([np.min(q1_range), np.min(q2_range), np.min(q3_range), np.min(q4_range)])
q_max = np.array([np.max(q1_range), np.max(q2_range), np.max(q3_range), np.max(q4_range)])
q_mean = (q_min + q_max) / 2
# Parametri
methods = ['inverse', 'redundant']
dt = 0.05
t_final = 10
times = np.arange(0, t_final + dt, dt)
total_time = np.concatenate([times, times + t_final])
k_values = [1, 5, 10, 20, 50, 100]
ka_values = [1, 5, 10, 50, 100]
best_err = float('inf')
best_k = None
best_ka = None
best_results = None
results = {}  # Dizionario esterno per tutti i risultati
for k in k_values:
    for ka in ka_values:
        print(f"\nTesting K = {k}, Ka = {ka}")
        for method in methods:
            q_curr = q0.copy()
            EE_traj = []
            DES_traj = []
            err_traj = []
            q_traj = []
            for t in times:
                pd, pdot = CircumferencePlanner(C1, pA, pB, 0, t_final, t, 'short')
                Qdot, _ = inv_kin(q_curr, pd, pdot, k, method, ka, dw_dq_func, q_mean, q_max, q_min)
                q_curr += Qdot * dt
                q_traj.append(q_curr.copy())
                _, _, _, p_ee, *_ = dir_kin(q_curr)
                err = np.linalg.norm(pd - p_ee)
                EE_traj.append(p_ee)
                DES_traj.append(pd)
                err_traj.append(err)
            for t in times:
                pd, pdot = CircumferencePlanner(C2, pB, pC, 0, t_final, t, 'short')
                Qdot, _ = inv_kin(q_curr, pd, pdot, k, method, ka, dw_dq_func, q_mean, q_max, q_min)
                q_curr += Qdot * dt
                q_traj.append(q_curr.copy())
                _, _, _, p_ee, *_ = dir_kin(q_curr)
                err = np.linalg.norm(pd - p_ee)
                EE_traj.append(p_ee)
                DES_traj.append(pd)
                err_traj.append(err)
            EE_traj = np.array(EE_traj)
            DES_traj = np.array(DES_traj)
            err_traj = np.array(err_traj)
            EE_vel = np.gradient(EE_traj, dt, axis=0)
            DES_vel = np.gradient(DES_traj, dt, axis=0)
            results[(k, ka, method)] = {
                'EE_traj': EE_traj,
                'DES_traj': DES_traj,
                'err_traj': err_traj,
                'EE_vel': EE_vel,
                'DES_vel': DES_vel,
                'q_traj': np.array(q_traj)
            }
            mean_err = np.mean(err_traj)
            print(f"  Metodo {method} -> Errore medio: {mean_err:.6f} m")
            if method == 'redundant' and mean_err < best_err:
                best_err = mean_err
                best_k = k
                best_ka = ka
                best_results = results[(k, ka, method)]
print("\n=== RISULTATO FINALE ===")
print(f"La miglior combinazione di parametri è: K = {best_k}, Ka = {best_ka}")
print(f"Errore medio corrispondente (metodo redundant): {best_err:.6f} m")
# Differenza tra metodi per la miglior combinazione trovata
diff = np.linalg.norm(results[(best_k, best_ka, 'inverse')]['EE_traj'] - best_results['EE_traj'], axis=1)
print("Differenza media tra inverse e redundant:", np.mean(diff))
print("Differenza massima:", np.max(diff))

Tb4 = 
 [[1.    0.    0.    0.16 ]
 [0.    1.    0.    0.03 ]
 [0.    0.    1.    0.105]
 [0.    0.    0.    1.   ]]

pee =  [0.16  0.03  0.105]
Point A has coordinates :  [0.16  0.03  0.105]
Point B has coordinates :  [0.1  0.15 0.13]
Point C has coordinates :  [-0.2   0.05  0.1 ]
 Workspace limits:
X: (np.float64(-0.24401149803226718), np.float64(0.24406006939314878))
Y: (np.float64(-0.05446518715458414), np.float64(0.1791962589453135))
Z: (np.float64(-0.10715424025921322), np.float64(0.2433422238502611))
All the points selected for the planning belong to the workspace
C1 [0.19538348 0.12269174 0.1175    ]
C2 [-0.13048292  0.34144875  0.115     ]

Testing K = 1, Ka = 1
  Metodo inverse -> Errore medio: 0.001269 m
  Metodo redundant -> Errore medio: 0.001249 m

Testing K = 1, Ka = 5
  Metodo inverse -> Errore medio: 0.001269 m
  Metodo redundant -> Errore medio: 0.001233 m

Testing K = 1, Ka = 10
  Metodo inverse -> Errore medio: 0.001269 m
  Metodo redundant -> Errore medio: 0.001204